# MambaVision Baseline: train -> predictions -> eval -> figures

Run this notebook in Kaggle with GPU enabled. It clones the project repository, builds or reuses standardized ACDC metadata, trains the dual-head MambaVision baseline, exports validation predictions, evaluates metrics, and packages outputs for the report.

## 1. Clone the project branch

Change `BRANCH` if the latest MambaVision code is pushed to another branch.

In [ ]:
REPO_URL = "https://github.com/Eroouu/DL_project.git"
BRANCH = "vasily/add-visualization"

!rm -rf /kaggle/working/DL_project
!git clone --branch {BRANCH} {REPO_URL} /kaggle/working/DL_project
%cd /kaggle/working/DL_project
!git status --short --branch

## 2. Install dependencies for MambaVision

Kaggle usually already has CUDA-enabled PyTorch. We keep PyTorch as provided by the runtime and install the project/runtime dependencies needed by Hugging Face MambaVision remote code.

In [ ]:
!pip install -q pandas numpy pillow scikit-learn matplotlib pyyaml transformers timm einops accelerate

# MambaVision remote code may require these packages. If installation fails, keep going
# and let the import/load check below show the exact missing dependency.
!pip install -q causal-conv1d mamba-ssm || true

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('gpu memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 3. Locate ACDC data and build metadata

Add the ACDC Kaggle dataset to the notebook input. The expected data root must contain:

- `rgb_anon_trainvaltest/rgb_anon/...`
- `gt_trainval/gt/...`

If auto-detection picks the wrong folder, set `DATA_ROOT` manually.

In [ ]:
from pathlib import Path

def looks_like_acdc_root(path: Path) -> bool:
    return (path / 'rgb_anon_trainvaltest' / 'rgb_anon').exists() and (path / 'gt_trainval' / 'gt').exists()

candidates = [p for p in Path('/kaggle/input').glob('**/*') if p.is_dir() and looks_like_acdc_root(p)]
print('ACDC root candidates:')
for candidate in candidates[:20]:
    print('-', candidate)

DATA_ROOT = candidates[0] if candidates else None
print('selected DATA_ROOT:', DATA_ROOT)
assert DATA_ROOT is not None, 'Set DATA_ROOT manually to the folder with rgb_anon_trainvaltest/ and gt_trainval/'

In [ ]:
!python -m src.dataset_builder \
  --data-root "{DATA_ROOT}" \
  --classmap configs/classmap.json \
  --out-dir /kaggle/working/metadata \
  --prefix metadata

!rm -rf ./metadata
!cp -r /kaggle/working/metadata ./metadata
!find metadata -maxdepth 1 -type f | sort

## 4. Validate metadata and image paths

In [ ]:
!python scripts/validate_data_contract.py --metadata-dir metadata --classmap configs/classmap.json

In [ ]:
import pandas as pd
from pathlib import Path

train_df = pd.read_csv('metadata/metadata_train.csv')
val_df = pd.read_csv('metadata/metadata_val.csv')
display(train_df.head())
print('train rows:', len(train_df), 'val rows:', len(val_df))
print('object columns:', len([c for c in train_df.columns if c.startswith('has_')]))
print('first image_path:', val_df.loc[0, 'image_path'])
print('first image exists:', Path(val_df.loc[0, 'image_path']).exists())

## 5. Eval pipeline check on real metadata with dummy predictions

In [ ]:
!python src/eval.py \
  --metadata metadata/metadata_val.csv \
  --dummy \
  --tune-thresholds \
  --out reports/metrics/metadata_val_dummy_eval.json

!python src/visualize.py \
  --metrics reports/metrics/metadata_val_dummy_eval.json \
  --out-dir reports/figures

## 6. Check that MambaVision-T loads

This isolates dependency/model-loading issues before the training command. The project model uses Hugging Face `AutoModel.from_pretrained(..., trust_remote_code=True)` and attaches the two classification heads from `src/models/mamba_dual_head.py`.

In [ ]:
from src.models import create_dual_head_model

model = create_dual_head_model(
    model_name='mamba_t',
    num_weather_classes=4,
    num_object_classes=18,
    hidden_dim=256,
    dropout=0.1,
    freeze_backbone=True,
)
print(type(model))
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Train MambaVision-T smoke baseline

This is a short run using `limit_train` / `limit_val` from `configs/mamba_t_smoke.yaml`.

In [ ]:
!python scripts/train.py \
  --config configs/mamba_t_smoke.yaml \
  --metadata-dir metadata \
  --output-dir checkpoints/mamba_t_smoke \
  --device auto

## 8. Export MambaVision predictions to NPZ

In [ ]:
!python scripts/predict.py \
  --checkpoint checkpoints/mamba_t_smoke/mamba_t_epoch01.pt \
  --metadata-dir metadata \
  --out artifacts/predictions/mamba_t_smoke_val.npz \
  --device auto

## 9. Evaluate MambaVision-T smoke baseline and build figures

In [ ]:
!python src/eval.py \
  --metadata metadata/metadata_val.csv \
  --predictions artifacts/predictions/mamba_t_smoke_val.npz \
  --class-map configs/classmap.json \
  --tune-thresholds \
  --out reports/metrics/mamba_t_smoke_val.json

!python src/visualize.py \
  --metrics reports/metrics/mamba_t_smoke_val.json \
  --out-dir reports/figures

## 10. Optional: full MambaVision-T run

Run this only after the smoke baseline works. If Kaggle GPU memory is tight, set `--batch-size 4`.

In [ ]:
# !python scripts/train.py \
#   --config configs/mamba_t_full.yaml \
#   --metadata-dir metadata \
#   --output-dir checkpoints/mamba_t_full \
#   --device auto \
#   --batch-size 8

# !python scripts/predict.py \
#   --checkpoint checkpoints/mamba_t_full/mamba_t_epoch05.pt \
#   --metadata-dir metadata \
#   --out artifacts/predictions/mamba_t_full_val.npz \
#   --device auto

# !python src/eval.py \
#   --metadata metadata/metadata_val.csv \
#   --predictions artifacts/predictions/mamba_t_full_val.npz \
#   --class-map configs/classmap.json \
#   --tune-thresholds \
#   --out reports/metrics/mamba_t_full_val.json

# !python src/visualize.py \
#   --metrics reports/metrics/mamba_t_full_val.json \
#   --out-dir reports/figures

## 11. Optional: compare against ResNet metrics

Upload or copy `reports/metrics/resnet50_val.json` into this runtime, then uncomment the comparison command.

In [ ]:
# !python scripts/compare_metrics.py \
#   --metrics reports/metrics/mamba_t_smoke_val.json reports/metrics/resnet50_val.json \
#   --names mamba_t_smoke resnet50 \
#   --out-csv reports/metrics/model_comparison.csv \
#   --out-md reports/metrics/model_comparison.md

## 12. Package outputs for report

In [ ]:
!zip -r mambavision_outputs.zip reports/metrics reports/figures artifacts/predictions checkpoints/mamba_t_smoke